<a href="https://colab.research.google.com/github/YaroslavTrusov/python-ai-template-Yaroslav_Trusov/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
- [`planes.csv`](https://github.com/YaroslavTrusov/python-ai-template-Yaroslav_Trusov/blob/main/data/planes.sparql) — виды самолетов, двигателей, цена, максимальная высота и суммарное количество, произведенных самолетов


**Что мы делаем:**
1. Клонируем репозиторий GitHub в Colab
2. Читаем CSV-файлы в pandas DataFrame
3. Очищаем и переименовываем столбцы
4. Смотрим структуру данных и делаем быструю валидацию

## 🐱 [1] Клонируем репозиторий курса в Colab

In [ ]:
# 🐱 Шаг 1. Клонируем репозиторий в Colab

import os

if not os.path.exists("python-ai-template-Yaroslav_Trusov"):
    !git clone -q https://github.com/YaroslavTrusov/python-ai-template-Yaroslav_Trusov

%cd python-ai-template-Yaroslav_Trusov

print("✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-template-Yaroslav_Trusov")

/content/python-ai-template-Yaroslav_Trusov/python-ai-template-Yaroslav_Trusov
✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-template-Yaroslav_Trusov


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем CSV-файл в объект `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в датасет.

In [ ]:
# 🐱 Шаг 2A. Чтение CSV-файла в pandas

import pandas as pd

df_planes = pd.read_csv("data/planes.csv")

print("✅ Загружено строк в df_planes:", len(df_planes))


✅ Загружено строк в df_planes: 870


## 🧹 [2B] Очистка и переименование столбцов

В исходных CSV-файлах есть **технические столбцы**, которые полезны для Викиданных, но мешают простому анализу:

- Столбец `aircraft` с URL (ссылкой на объект Wikidata) — **сохраняем его для отладки**, но переименуем в `aircraftURL`.
- Столбцы `aircraftLabel`, `engine`, `engineLabel`, `maxAltitude`, `price`, `totalProduced` содержат читаемые подписи (названия).

В этом шаге мы:
- переименуем столбец с URL Wikidata (`aircraft` → `aircraftURL`);
- переименуем `aircraftLabel → aircraft`, `engine → engineURL`, `engineLabel → engine`, `maxAltitude → altitude`, `totalProduced → produced`;
- приведём числовые столбцы (`altitude`,`price` `produced`) к типу `int`.

При приведении к числам мы используем:

- `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные значения превращает в `NaN`;
- `fillna(0)` — заменяет пропущенные значения (`NaN`) на 0;
- `astype(int)` — переводит столбец к целочисленному типу.

> ⚠️ **Важно:** если в ваших данных есть столбцы с URL Wikidata и столбцы вида `*Label`, этот шаг обязателен, чтобы получить аккуратные таблички для анализа. Столбец `URL` пригодится, если нужно будет быстро перейти к оригинальной записи в Викиданных.

In [ ]:
# 🧹 Шаг 2B. Очистка и переименование столбцов

# 1) df_planes: виды самолетов, двигателей, цена, максимальная высота и суммарное количество, произведенных самолетов
if "aircraftLabel" in df_planes.columns:                             # Проверка: нужна ли очистка?
    # URL Wikidata не удаляем, а переименовываем в "URL" для удобства
    df_planes = df_planes.rename(columns={
        "aircraft": "aircraftURL",          # ← переименовали технический столбец
        "aircraftLabel": "aircraft",
        "engine": "engineURL",
        "engineLabel": "engine",
        "maxAltitude": "altitude",
        "totalProduced": "produced",
    })
if df_planes["altitude"].dtype != 'int64':
    df_planes["altitude"] = pd.to_numeric(
        df_planes["altitude"], errors="coerce"
    ).fillna(0).astype(int)
    print("✅ df_planes очищен")
else:
    print("⏭️ df_planes уже очищен, пропускаем")

if df_planes["price"].dtype != 'int64':
    df_planes["price"] = pd.to_numeric(
        df_planes["price"], errors="coerce"
    ).fillna(0).astype(int)
    print("✅ df_planes очищен")
else:
    print("⏭️ df_planes уже очищен, пропускаем")

if df_planes["produced"].dtype != 'int64':
    df_planes["produced"] = pd.to_numeric(
        df_planes["produced"], errors="coerce"
    ).fillna(0).astype(int)
    print("✅ df_planes очищен")
else:
    print("⏭️ df_planes уже очищен, пропускаем")

print("\n✅ Данные готовы к анализу")

✅ df_planes очищен
✅ df_planes очищен
⏭️ df_planes уже очищен, пропускаем

✅ Данные готовы к анализу


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор обоих DataFrame:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк.

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы не повторять один и тот же код два раза.

In [ ]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))
    print("\nПервые строки:")
    print(df.head(n))

# 🔍 Шаг 3. Обзор данных

show_info(df_planes, "Виды самолетов, двигателей, цена, максимальная высота и суммарное количество, произведенных самолетов (df_planes)")



📊 Виды самолетов, двигателей, цена, максимальная высота и суммарное количество, произведенных самолетов (df_planes)
Размер: (870, 7)
Столбцы: aircraftURL, aircraft, engineURL, engine, altitude, price, produced

Первые строки:
                              aircraftURL            aircraft  \
0  http://www.wikidata.org/entity/Q211443                Ил-2   
1  http://www.wikidata.org/entity/Q943360                 У-2   
2  http://www.wikidata.org/entity/Q207700        Антонов Ан-2   
3  http://www.wikidata.org/entity/Q562475                Як-9   
4  http://www.wikidata.org/entity/Q154106  Bell UH-1 Iroquois   

                                  engineURL             engine  altitude  \
0    http://www.wikidata.org/entity/Q599854              АМ-38         0   
1    http://www.wikidata.org/entity/Q742981               М-11         0   
2  http://www.wikidata.org/entity/Q15051031  Shvetsov ASh-62IR         0   
3    http://www.wikidata.org/entity/Q977265              М-105     10750   
4 

## ✅ [4] Быстрая проверка и валидация данных

Здесь мы посмотрим:

- сколько **уникальных** самолетов, двигателей есть в данных;
- **какие самолеты встречаются чаще всего** (Топ‑5 по числу строк);
- **какие двигатели самые популярные** (Топ‑10 по числу строк).

Функция `value_counts()`:
- считает, сколько раз каждое значение встречается в столбце;
- сортирует результаты по убыванию.

Метод `.head()` берёт первые N строк, поэтому
`df_planes["aircraft"].value_counts().head()` даёт **Топ‑5 самолетов по числу записей**.

In [ ]:
# ✅ Шаг 4. Быстрая проверка и валидация данных

print("🔍 Быстрая проверка данных")

# Датасет 1: Виды самолетов, двигателей, цена, максимальная высота и суммарное количество, произведенных самолетов
print("\n📊 Датасет: Виды самолетов, двигателей, цена, максимальная высота и суммарное количество, произведенных самолетов")
print("Уникальных самолетов в df_planes:", df_planes["aircraft"].nunique())
print("Уникальных двигателей:", df_planes["engine"].nunique())


print("\nТоп-5 стран по числу записей:")
print(df_planes["aircraft"].value_counts().head())

print("\nТоп-10 жанров:")
print(df_planes["engine"].value_counts().head(10))


🔍 Быстрая проверка данных

📊 Датасет: Виды самолетов, двигателей, цена, максимальная высота и суммарное количество, произведенных самолетов
Уникальных самолетов в df_planes: 522
Уникальных двигателей: 390

Топ-5 стран по числу записей:
aircraft
Airbus A320       16
Dietrich DP.II    16
МС-21             14
DKW Erla Me 5a    12
Bartel BM-5       12
Name: count, dtype: int64

Топ-10 жанров:
engine
BMW VI                     17
Siemens & Halske Sh 12     15
Pratt & Whitney PW1000G    13
Siemens-Halske Sh 11       12
Walter Minor 4             11
Benz Bz.IV                 10
PowerJet SaM146             8
Siemens & Halske Sh 14      8
ПД-14                       7
Walter Minor 4-III          7
Name: count, dtype: int64


## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали CSV-файл из `data/`
- ✅ Удалили URL Wikidata и переименовали столбцы (`*Label → короткие имена`)
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Выполнили быструю валидацию:
  - количество уникальных самолетов, двигателей
  - диапазоны значений
  - топ самолетов и двигателей по числу записей
  

Теперь у нас есть **аккуратные, проверенные таблицы**, с которыми удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **те же данные** для:
- более сложного анализа (группировки, фильтрация),
- и построения визуализаций (графики и диаграммы). 🎨